# 03 Preprocessing — WLC-Eignungsfaktoren

**PA1 ZHAW IUNR** | Bächler, Hofstetter, Reichlin | Betreuer: Patrick Laube

Prüft und verarbeitet alle Datensätze für die Weighted Linear Combination.

**Einziger Input aus vorherigen Notebooks:**
```
data/processed/constraints/constraint_mask_s2.tif
```
Dieses Notebook ist unabhängig von 01/02a ausführbar — eine beliebige binäre Maske (1 = geeignet, 0 = ausgeschlossen) reicht als Input.

**Output:** 10 normalisierte Raster + Akzeptanz-Layer + Distanz-Raster

## 1. Setup

In [ ]:
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd
import pickle
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from rasterio.warp import reproject
from scipy.ndimage import distance_transform_edt
import fiona
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

RAW = Path("../data/raw")
PROC = Path("../data/processed")
OUT = Path("../outputs/figures")
for d in [PROC / "criteria", PROC / "distances", OUT]:
    d.mkdir(parents=True, exist_ok=True)

CRS = "EPSG:2056"
RES = 25
NODATA = -9999.0

## 2. Constraint-Maske & Referenzraster laden

In [ ]:
mask_path = PROC / "constraints/constraint_mask_s2.tif"
with rasterio.open(mask_path) as src:
    constraint_mask = src.read(1)
    transform = src.transform
    ref_profile = src.profile.copy()
    ref_profile.update(dtype="float32", nodata=NODATA)
    height, width = constraint_mask.shape

# DEM (aus 02a)
with rasterio.open(PROC / "dem/dem_gr_25m.tif") as src:
    dem = src.read(1)
with rasterio.open(PROC / "dem/slope_gr_25m.tif") as src:
    slope = src.read(1)
with rasterio.open(PROC / "dem/aspect_gr_25m.tif") as src:
    aspect = src.read(1)

valid = (dem != NODATA) & (constraint_mask == 1)
print(f"Constraint-Maske: {constraint_mask.sum():,} Pixel ({constraint_mask.sum()*RES*RES/1e6:.0f} km²)")
print(f"Raster: {width}×{height} @ {RES}m")

In [ ]:
# Hilfsfunktionen

def normalize_linear(data, low, high, invert=False):
    out = np.full_like(data, NODATA)
    m = valid & (data != NODATA)
    vals = np.clip((data[m] - low) / (high - low), 0, 1)
    out[m] = 1 - vals if invert else vals
    return out

def normalize_piecewise(data, breakpoints, values):
    out = np.full_like(data, NODATA)
    m = valid & (data != NODATA)
    out[m] = np.interp(data[m], breakpoints, values)
    return out

def save_criterion(data, name):
    path = PROC / f"criteria/{name}.tif"
    with rasterio.open(path, "w", **ref_profile) as dst:
        dst.write(data, 1)
    v = data[valid & (data != NODATA)]
    print(f"  {name}: {v.min():.3f}–{v.max():.3f} (mean {v.mean():.3f})")

def rasterize_to_grid(gdf, burn=1):
    shapes = [(g, burn) for g in gdf.to_crs(CRS).geometry if g is not None]
    if not shapes:
        return np.zeros((height, width), dtype=np.uint8)
    return rasterize(shapes, out_shape=(height, width), transform=transform, fill=0, dtype=np.uint8)

def compute_distance(binary_raster, name):
    dist = distance_transform_edt((binary_raster == 0).astype(float), sampling=(RES, RES)).astype(np.float32)
    with rasterio.open(PROC / f"distances/{name}.tif", "w", **ref_profile) as dst:
        dst.write(dist, 1)
    print(f"  {name}: max {dist[valid].max():.0f} m")
    return dist

## 3. Datenprüfung — WLC-Datensätze

Qualitätskontrolle der Datensätze die NUR für WLC relevant sind (nicht für Constraints).

In [ ]:
print("=== WLC-Datensätze prüfen ===\n")

# BFE Solar
for deg in ["0", "30", "75", "90"]:
    parent = RAW / "solar" / f"solarenergie-einstrahlung_{deg}_grad_2056.gpkg"
    gpkg = next(parent.glob("*.gpkg"), None) if parent.is_dir() else (parent if parent.exists() else None)
    if gpkg:
        gdf = gpd.read_file(gpkg, rows=slice(0, 1))
        print(f"BFE Solar {deg}°: ✓ CRS={gdf.crs}")
    else:
        print(f"BFE Solar {deg}°: ⚠ FEHLT")

# SPASS Schnee
spass = sorted((RAW / "snow").glob("**/*.nc"))
print(f"SPASS Schnee: {'✓ ' + spass[0].name if spass else '⚠ FEHLT'}")

# BFE Stromnetz
netz = sorted((RAW / "infrastructure/power_grid").glob("*.shp")) + \
       sorted((RAW / "infrastructure/power_grid").glob("*.gpkg"))
print(f"BFE Stromnetz: {'✓ ' + netz[0].name if netz else '⚠ FEHLT'}")

# KlG Akzeptanz
klg = RAW / "acceptance/klg_2023.json"
print(f"KlG Akzeptanz: {'✓' if klg.exists() else '⚠ FEHLT'}")

# swissTLM3D (Strassen, Staumauern — WLC-spezifisch)
tlm_path = RAW / "tlm3d/swisstlm3d_2026-02-24_2056_5728.gpkg"
if tlm_path.is_dir():
    tlm_path = next(tlm_path.glob("*.gpkg"))
print(f"swissTLM3D: {'✓' if tlm_path.exists() else '⚠ FEHLT'}")

## 4. BFE Solar → Raster

In [ ]:
print("=== F01: Globalstrahlung ===")
solar_parent = RAW / "solar/solarenergie-einstrahlung_0_grad_2056.gpkg"
gpkg = next(solar_parent.glob("*.gpkg")) if solar_parent.is_dir() else solar_parent

bbox = (transform.c, transform.f + height * transform.e,
        transform.c + width * transform.a, transform.f)
gdf_solar = gpd.read_file(gpkg, bbox=bbox)
num_col = [c for c in gdf_solar.select_dtypes(include="number").columns if c != "geometry"][0]
print(f"  Spalte: {num_col} | Features: {len(gdf_solar)} | {gdf_solar[num_col].min():.0f}–{gdf_solar[num_col].max():.0f} kWh/m²/a")

shapes = [(g, v) for g, v in zip(gdf_solar.geometry, gdf_solar[num_col]) if g is not None and not np.isnan(v)]
solar_raster = rasterize(shapes, out_shape=(height, width), transform=transform, fill=NODATA, dtype=np.float32)
v = solar_raster[valid & (solar_raster != NODATA)]
f01 = normalize_linear(solar_raster, v.min(), v.max())
save_criterion(f01, "f01_globalstrahlung")

In [ ]:
print("\n=== F02: Wintereinstrahlung (75° als Proxy) ===")
solar_w = RAW / "solar/solarenergie-einstrahlung_75_grad_2056.gpkg"
gpkg_w = next(solar_w.glob("*.gpkg")) if solar_w.is_dir() else solar_w
gdf_w = gpd.read_file(gpkg_w, bbox=bbox)
num_col_w = [c for c in gdf_w.select_dtypes(include="number").columns if c != "geometry"][0]
shapes_w = [(g, v) for g, v in zip(gdf_w.geometry, gdf_w[num_col_w]) if g is not None and not np.isnan(v)]
winter_raster = rasterize(shapes_w, out_shape=(height, width), transform=transform, fill=NODATA, dtype=np.float32)
v_w = winter_raster[valid & (winter_raster != NODATA)]
f02 = normalize_linear(winter_raster, v_w.min(), v_w.max())
save_criterion(f02, "f02_wintereinstrahlung")

## 5. DEM-basierte Eignungsfaktoren

In [ ]:
print("=== F03: Hangneigung (Optimum 20–30°) ===")
f03 = normalize_piecewise(slope, [0, 20, 30, 45, 90], [0, 1, 1, 0, 0])
save_criterion(f03, "f03_hangneigung")

print("\n=== F04: Exposition (Südabweichung) ===")
south_dev = np.abs(aspect - 180)
f04 = normalize_linear(south_dev, 180, 0)
save_criterion(f04, "f04_exposition")

print("\n=== F05: Höhenlage (Optimum 1800–2500m) ===")
f05 = normalize_piecewise(dem, [1500, 1800, 2500, 2700], [0, 1, 1, 0])
save_criterion(f05, "f05_hoehenlage")

## 6. SPASS Schneebedeckung

In [ ]:
print("=== F06: Schneebedeckung (SPASS) ===")
spass_files = sorted((RAW / "snow").glob("**/*.nc"))
if not spass_files:
    print("  ⚠ SPASS nicht gefunden — F06 übersprungen")
else:
    import xarray as xr
    ds = xr.open_dataset(spass_files[0], engine="h5netcdf", decode_times=False)
    swe_var = next((v for v in ds.data_vars if "swe" in v.lower()), list(ds.data_vars)[0])
    print(f"  Datei: {spass_files[0].name} | Var: {swe_var} | Dims: {dict(ds.dims)}")

    snow_frac = (ds[swe_var] > 0).mean(dim="time").values.astype(np.float32)

    if "E" in ds.dims:
        x, y = ds["E"].values, ds["N"].values
    elif "chx" in ds.dims:
        x, y = ds["chx"].values, ds["chy"].values
    else:
        x, y = ds[list(ds.dims)[1]].values, ds[list(ds.dims)[0]].values

    src_tf = from_bounds(x.min(), y.min(), x.max(), y.max(), len(x), len(y))
    snow_25m = np.full((height, width), NODATA, dtype=np.float32)
    reproject(source=np.flipud(snow_frac), destination=snow_25m,
              src_transform=src_tf, src_crs=CRS, dst_transform=transform, dst_crs=CRS,
              resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=NODATA)

    v_s = snow_25m[valid & (snow_25m != NODATA)]
    f06 = normalize_linear(snow_25m, v_s.min(), v_s.max())
    save_criterion(f06, "f06_schneebedeckung")
    ds.close()

## 7. Distanz-Raster (Strasse, Netz, Infrastruktur)

In [ ]:
print("=== Distanz-Raster ===\n")

# Strassen aus swissTLM3D
strassen = gpd.read_file(str(tlm_path), layer="tlm_strassen_strasse", bbox=bbox)
print(f"  Strassen: {len(strassen)} Features")
strassen_raster = rasterize_to_grid(strassen)
dist_strasse = compute_distance(strassen_raster, "dist_strasse_gr_25m")

# BFE Stromnetz
netz_files = sorted((RAW / "infrastructure/power_grid").glob("*.shp")) + \
             sorted((RAW / "infrastructure/power_grid").glob("*.gpkg"))
if netz_files:
    netz = gpd.read_file(netz_files[0]).to_crs(CRS)
    netz_raster = rasterize_to_grid(netz)
    dist_netz = compute_distance(netz_raster, "dist_netz_gr_25m")
else:
    print("  ⚠ Stromnetz nicht gefunden")
    dist_netz = np.full((height, width), 5000, dtype=np.float32)

# Staumauern aus swissTLM3D
try:
    staumauern = gpd.read_file(str(tlm_path), layer="tlm_bauten_staubaute", bbox=bbox)
    print(f"  Staumauern: {len(staumauern)} Features")
    infra_raster = rasterize_to_grid(staumauern)
    dist_infra = compute_distance(infra_raster, "dist_infrastruktur_gr_25m")
except:
    print("  ⚠ Staumauer-Layer nicht gefunden")
    dist_infra = np.full((height, width), 10000, dtype=np.float32)

# Normalisieren (inverse: kürzer = besser)
for name, dist, fid in [("netzanschluss", dist_netz, "f07"),
                         ("strasse", dist_strasse, "f08"),
                         ("infrastruktur", dist_infra, "f09")]:
    v_d = dist[valid & (dist != NODATA)]
    p95 = np.percentile(v_d, 95) if v_d.size else 10000
    f = normalize_linear(dist, 0, p95, invert=True)
    save_criterion(f, f"{fid}_{name}")

## 8. Sichtbarkeit (Proxy)

In [ ]:
print("=== F10: Sichtbarkeit ===")
siedl_path = PROC / "constraints/c13_siedlung.tif"
if siedl_path.exists():
    with rasterio.open(siedl_path) as src:
        siedl = src.read(1)
    dist_siedlung = distance_transform_edt((siedl == 0).astype(float), sampling=(RES, RES)).astype(np.float32)
    v_s = dist_siedlung[valid]
    p95 = np.percentile(v_s, 95) if v_s.size else 5000
    f10 = normalize_linear(dist_siedlung, 0, p95)
    save_criterion(f10, "f10_sichtbarkeit")
else:
    print("  ⚠ Siedlungs-Maske nicht gefunden")

## 9. KlG-Akzeptanz → Gemeinde-Raster

In [ ]:
print("=== Akzeptanz-Layer ===")
muni = gpd.read_file(RAW / "swissboundaries/swissboundaries_gemeinden.shp")
canton_col = next(c for c in muni.columns if "KANT" in c.upper() or c.upper().startswith("KT"))
name_col = next(c for c in muni.columns if "NAME" in c.upper())
gr_muni = muni[muni[canton_col].astype(str).str.contains("GR", case=False, na=False)].copy()

df_klg = pd.read_json(RAW / "acceptance/klg_2023.json")
df_klg["yes_share"] = df_klg["yeas"] / (df_klg["yeas"] + df_klg["nays"])
print(f"  KlG: {len(df_klg)} Gemeinden, Ja-Anteil {df_klg['yes_share'].min():.2f}–{df_klg['yes_share'].max():.2f}")

gr_muni["_k"] = gr_muni[name_col].str.strip().str.upper()
votes = df_klg.assign(_k=df_klg["name"].str.strip().str.upper())
gdf_acc = gr_muni.merge(votes[["_k", "yes_share"]], on="_k", how="left")
gdf_acc["yes_share"] = gdf_acc["yes_share"].fillna(gdf_acc["yes_share"].median())
print(f"  Join: {gdf_acc['yes_share'].notna().sum()}/{len(gdf_acc)} Gemeinden")

shapes = [(g, v) for g, v in zip(gdf_acc.geometry, gdf_acc["yes_share"]) if g is not None]
acc = rasterize(shapes, out_shape=(height, width), transform=transform, fill=NODATA, dtype=np.float32)
with rasterio.open(PROC / "criteria/acceptance_klg_gr_25m.tif", "w", **ref_profile) as dst:
    dst.write(acc, 1)
v_a = acc[valid & (acc != NODATA)]
print(f"  Akzeptanz: {v_a.min():.2f}–{v_a.max():.2f}")

## 10. Übersicht & Vorschau

In [ ]:
print("=== Erzeugte Dateien ===\n")
for folder in ["criteria", "distances"]:
    files = sorted((PROC / folder).glob("*.tif"))
    if files:
        print(f"{folder}/")
        for f in files:
            print(f"  {f.name:40s} {f.stat().st_size/1e6:>6.1f} MB")
        print()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
fig.suptitle("Normalisierte Eignungsfaktoren [0, 1]", fontsize=13, fontweight="bold")
criteria = sorted((PROC / "criteria").glob("f*.tif"))
for ax, fp in zip(axes.flat, criteria):
    with rasterio.open(fp) as src:
        data = np.ma.masked_equal(src.read(1), NODATA)
    ax.imshow(data, cmap="YlOrRd", vmin=0, vmax=1, aspect="equal")
    ax.set_title(fp.stem, fontsize=8); ax.set_axis_off()
for ax in axes.flat[len(criteria):]:
    ax.set_visible(False)
plt.tight_layout()
fig.savefig(OUT / "wlc_criteria_overview.png", dpi=150, bbox_inches="tight")

## Nächster Schritt

Alle Eignungsfaktoren liegen als normalisierte GeoTIFFs in `data/processed/criteria/` vor.

**→ Weiter mit:** WLC-Verschneidung (AHP-Gewichte × Kriterien × Constraint-Maske × Akzeptanz)

**Interface-Garantie:** Dieses Notebook benötigt aus 01/02a nur `constraint_mask_s2.tif` und die DEM-Ableitungen. Es kann mit einer beliebigen binären Maske betrieben werden.